# NYC Taxi Demand Forecast

Hourly yellow-taxi pickups across New York City, forecast one week ahead per borough and compared across three models with rolling-origin backtests. Data: [NYC TLC trip records](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) + [Open-Meteo](https://open-meteo.com/) historical weather, loaded into MySQL by this repo's pipeline. Source: [wirkix/nyc-taxi-demand-forecast](https://github.com/wirkix/nyc-taxi-demand-forecast).

In [ ]:
# Parameters (overridden by papermill)
run_id = None  # None = latest run

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sqlalchemy as sa
from IPython.display import HTML, Markdown, display

from load.db import get_engine

engine = get_engine()
MODEL_LABELS = {
    "seasonal_naive": "Seasonal naive (last week)",
    "gradient_boosting": "Gradient boosting",
    "prophet": "Prophet",
}


def show(fig):
    # Inline HTML with plotly.js from the CDN: renders in the nbconvert
    # HTML export too, unlike the default renderer (needs require.js).
    fig.update_layout(template="plotly_white", margin=dict(l=40, r=20, t=60, b=40))
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


def query(sql, **params):
    with engine.connect() as conn:
        return pd.read_sql(sa.text(sql), conn, params=params)


if run_id is None:
    run_id = query("SELECT run_id FROM forecast_runs ORDER BY created_at DESC LIMIT 1")[
        "run_id"
    ].iloc[0]
run = query("SELECT * FROM forecast_runs WHERE run_id = :r", r=run_id).iloc[0]
display(
    Markdown(
        f"**Run** `{run_id}` · trained through **{run.train_end:%Y-%m-%d %H:%M}** · "
        f"{run.horizon_hours}h horizon · {run.n_folds} backtest folds · "
        f"generated {run.created_at:%Y-%m-%d}"
    )
)

## Demand history

Daily pickups by borough, with the seasonal shape the models have to learn.

In [ ]:
daily = query(
    "SELECT DATE(pickup_hour) AS day, borough, SUM(trips) AS trips "
    "FROM v_hourly_pickups_borough GROUP BY DATE(pickup_hour), borough"
)
# GROUP BY output is unordered; unsorted x makes px.line zig-zag.
daily = daily[daily["borough"].isin(["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"])]
daily = daily.sort_values(["borough", "day"])
show(
    px.line(
        daily,
        x="day",
        y="trips",
        color="borough",
        log_y=True,
        title="Daily yellow-taxi pickups by borough (log scale)",
        labels={"day": "", "trips": "pickups/day", "borough": ""},
    )
)

In [ ]:
hourly = query(
    "SELECT pickup_hour, SUM(trips) AS trips FROM v_hourly_pickups_borough GROUP BY pickup_hour"
).sort_values("pickup_hour")
hourly["dow"] = hourly["pickup_hour"].dt.day_name()
hourly["hour"] = hourly["pickup_hour"].dt.hour
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
grid = hourly.pivot_table(index="dow", columns="hour", values="trips", aggfunc="mean").reindex(
    order
)
show(
    px.imshow(
        grid,
        aspect="auto",
        color_continuous_scale="Blues",
        title="Average pickups by hour of day and weekday (all boroughs)",
        labels={"x": "hour of day", "y": "", "color": "pickups/hour"},
    )
)

## Does rain move demand?

Average pickups per hour of day, split by whether it was raining in that hour (Central Park, > 0.5 mm). This is the weather signal the gradient-boosting and Prophet models get as features.

In [ ]:
weather = query("SELECT obs_hour, precipitation_mm, temperature_c FROM weather_hourly")
wx = hourly.merge(weather, left_on="pickup_hour", right_on="obs_hour")
wx["condition"] = (wx["precipitation_mm"] > 0.5).map({True: "Raining", False: "Dry"})
weekday = wx[wx["pickup_hour"].dt.dayofweek < 5]
rain = weekday.groupby(["hour", "condition"], as_index=False)["trips"].mean()
show(
    px.line(
        rain,
        x="hour",
        y="trips",
        color="condition",
        markers=True,
        title="Weekday pickups per hour: raining vs dry hours",
        labels={"hour": "hour of day", "trips": "avg pickups/hour", "condition": ""},
    )
)

## Backtests

Each fold trains on everything before a cutoff and forecasts the following week; folds are the most recent non-overlapping weeks. **WAPE** = total absolute error ÷ total actual trips (lower is better). A model only earns its complexity if it beats the seasonal-naive baseline.

In [ ]:
metrics = query("SELECT * FROM backtest_metrics WHERE run_id = :r", r=run_id)
metrics["model"] = metrics["model"].map(MODEL_LABELS)
summary = metrics.groupby(["series_id", "model"], as_index=False)[["wape", "mae", "rmse"]].mean()
show(
    px.bar(
        summary,
        x="series_id",
        y="wape",
        color="model",
        barmode="group",
        title="Mean backtest WAPE by series and model (lower is better)",
        labels={"series_id": "", "wape": "WAPE", "model": ""},
    ).update_yaxes(tickformat=".0%")
)
table = summary.pivot(index="series_id", columns="model", values="wape")
display(table.style.format("{:.1%}").highlight_min(axis=1, color="#d6f5e3"))

In [ ]:
bt = query(
    "SELECT * FROM backtest_predictions WHERE run_id = :r AND series_id = 'total' "
    "AND fold = (SELECT MAX(fold) FROM backtest_predictions WHERE run_id = :r)",
    r=run_id,
).sort_values("target_hour")
fig = go.Figure()
actual = bt.drop_duplicates("target_hour")
fig.add_scatter(
    x=actual["target_hour"], y=actual["y"], name="Actual", line=dict(color="black", width=2)
)
for model, part in bt.groupby("model"):
    fig.add_scatter(
        x=part["target_hour"], y=part["yhat"], name=MODEL_LABELS[model], line=dict(width=1.5)
    )
fig.update_layout(
    title="Most recent backtest week: forecast vs actual (all boroughs)", yaxis_title="pickups/hour"
)
show(fig)

## Next-week forecast

TLC publishes trip data about two months behind, so the "future" week here is the one right after the latest published month. It gets scored against actuals once the next month is released.

In [ ]:
fc = query("SELECT * FROM forecast_predictions WHERE run_id = :r AND series_id = 'total'", r=run_id)
fc = fc.sort_values("target_hour")
recent = hourly[hourly["pickup_hour"] > run.train_end - pd.Timedelta(days=14)]
fig = go.Figure()
fig.add_scatter(
    x=recent["pickup_hour"],
    y=recent["trips"],
    name="Actual (last 2 weeks)",
    line=dict(color="black", width=2),
)
for model, part in fc.groupby("model"):
    fig.add_scatter(
        x=part["target_hour"], y=part["yhat"], name=MODEL_LABELS[model], line=dict(width=1.5)
    )
fig.add_vline(x=run.train_end, line_dash="dot", line_color="gray")
fig.update_layout(title="Week-ahead forecast, all boroughs", yaxis_title="pickups/hour")
show(fig)